# Project Aftershock — Phase 2: Data Understanding (Native EDA)

## 1. Importing the needed libraries (modules)

In [14]:
import json
import csv
import requests
from pathlib import Path

## 2. Define Project Paths

In [15]:
project_root = Path("..")

raw_data = project_root / "data" / "raw"
processed_data = project_root / "data" / "processed"

raw_data.mkdir(parents=True, exist_ok=True)
processed_data.mkdir(parents=True, exist_ok=True)

## 3. Fetch Data from the API

In [16]:
url = "https://earthquake.usgs.gov/fdsnws/event/1/query"

params = {
    "format": "geojson",
    "starttime": "2026-08-01",
    "endtime": "2026-08-25",
    "minmagnitude": 2.5
}

response = requests.get(url, params=params, timeout=20)

print(f"status_code: {response.status_code}")

status_code: 200


In [17]:
if response.status_code == 200:
    print("API request successful.")
else:
    print("API request failed.")
    print("Status code:", response.status_code)

API request successful.


In [18]:
data = response.json()

print(f"Top-level data type: {type(data)}")
print(f"Top-level keys: {data.keys()}")

Top-level data type: <class 'dict'>
Top-level keys: dict_keys(['type', 'metadata', 'features', 'bbox'])


## 4. Check the API response structure

In [19]:
print("Top-level structure: ")
for key, value in data.items():
    print(f"{key}: {type(value).__name__}")

Top-level structure: 
type: str
metadata: dict
features: list
bbox: list


In [20]:
features = data['features']
records = features

print(f"Number of records: {len(features)}")

Number of records: 1780


In [21]:
# inspecting the 1st features record
first_record = features[0]

print(json.dumps(first_record, indent=4))

{
    "type": "Feature",
    "properties": {
        "mag": 4.6,
        "place": "103 km SSE of Tarama, Japan",
        "time": 1787614016418,
        "updated": 1787700337040,
        "tz": null,
        "url": "https://earthquake.usgs.gov/earthquakes/eventpage/us6000tn9r",
        "detail": "https://earthquake.usgs.gov/fdsnws/event/1/query?eventid=us6000tn9r&format=geojson",
        "felt": null,
        "cdi": null,
        "mmi": null,
        "alert": null,
        "status": "reviewed",
        "tsunami": 0,
        "sig": 326,
        "net": "us",
        "code": "6000tn9r",
        "ids": ",us6000tn9r,",
        "sources": ",us,",
        "types": ",origin,phase-data,",
        "nst": 25,
        "dmin": 0.919,
        "rms": 0.65,
        "gap": 128,
        "magType": "mb",
        "type": "earthquake",
        "title": "M 4.6 - 103 km SSE of Tarama, Japan"
    },
    "geometry": {
        "type": "Point",
        "coordinates": [
            124.9935,
            23.7741,
  

## 5. Recursive Structural Audit

In [22]:
def audit_structure(obj, path="root"):

    if isinstance(obj, dict):
        print(f"{path} -> dict")

        for key, value in obj.items():
            audit_structure(value, f"{path}.{key}")

    elif isinstance(obj, list):
        print(f"{path} -> list")

        for index, value in enumerate(obj):
            audit_structure(value, f"{path}[{index}]")

    else:
        print(f"{path} -> {type(obj).__name__}")

In [24]:
audit_structure(first_record)

root -> dict
root.type -> str
root.properties -> dict
root.properties.mag -> float
root.properties.place -> str
root.properties.time -> int
root.properties.updated -> int
root.properties.tz -> NoneType
root.properties.url -> str
root.properties.detail -> str
root.properties.felt -> NoneType
root.properties.cdi -> NoneType
root.properties.mmi -> NoneType
root.properties.alert -> NoneType
root.properties.status -> str
root.properties.tsunami -> int
root.properties.sig -> int
root.properties.net -> str
root.properties.code -> str
root.properties.ids -> str
root.properties.sources -> str
root.properties.types -> str
root.properties.nst -> int
root.properties.dmin -> float
root.properties.rms -> float
root.properties.gap -> int
root.properties.magType -> str
root.properties.type -> str
root.properties.title -> str
root.geometry -> dict
root.geometry.type -> str
root.geometry.coordinates -> list
root.geometry.coordinates[0] -> float
root.geometry.coordinates[1] -> float
root.geometry.coordin

## 8. Inspect important fields

In [25]:
sample = records[0]

print("Event ID:", sample.get("id"))

properties = sample.get("properties", {})
geometry = sample.get("geometry", {})

print("\nProperties:")
for key in properties:
    print("-", key)

print("\nGeometry:")
print("Type:", geometry.get("type"))
print("Coordinates:", geometry.get("coordinates"))

Event ID: us6000tn9r

Properties:
- mag
- place
- time
- updated
- tz
- url
- detail
- felt
- cdi
- mmi
- alert
- status
- tsunami
- sig
- net
- code
- ids
- sources
- types
- nst
- dmin
- rms
- gap
- magType
- type
- title

Geometry:
Type: Point
Coordinates: [124.9935, 23.7741, 10]


## 9. Exploratory Data Analysis (EDA)

In [26]:
magnitude_values = []

for record in records:
    magnitude = record.get("properties", {}).get("mag")
    
    if magnitude is not None:
        magnitude_values.append(float(magnitude))

print(f"Number of valid magnitude values: {len(magnitude_values)}")

Number of valid magnitude values: 1780


In [27]:
magnitude_min = None
magnitude_max = None
magnitude_total = 0
magnitude_count = 0

for magnitude in magnitude_values:
    magnitude_total += magnitude
    magnitude_count += 1
    
    if magnitude_min is None or magnitude < magnitude_min:
        magnitude_min = magnitude
    
    if magnitude_max is None or magnitude > magnitude_max:
        magnitude_max = magnitude

magnitude_mean = magnitude_total / magnitude_count

print(f"Minimum magnitude: {magnitude_min}")
print(f"Maximum magnitude: {magnitude_max}")
print(f"Mean magnitude: {magnitude_mean}")

Minimum magnitude: 2.5
Maximum magnitude: 7.7
Mean magnitude: 3.732887847226335


In [28]:
depth_values = []

for record in records:
    coordinates = record.get("geometry", {}).get("coordinates", [])
    
    if len(coordinates) >= 3:
        depth = coordinates[2]
        
        if depth is not None:
            depth_values.append(float(depth))

print(f"Number of valid depth values: {len(depth_values)}")

Number of valid depth values: 1780


In [29]:
depth_min = None
depth_max = None
depth_total = 0
depth_count = 0

for depth in depth_values:
    depth_total += depth
    depth_count += 1
    
    if depth_min is None or depth < depth_min:
        depth_min = depth
    
    if depth_max is None or depth > depth_max:
        depth_max = depth

depth_mean = depth_total / depth_count

print(f"Minimum depth (km): {depth_min}")
print(f"Maximum depth (km): {depth_max}")
print(f"Mean depth (km): {depth_mean}")

Minimum depth (km): -2.84
Maximum depth (km): 665.326
Mean depth (km): 48.65573432513952


In [30]:
fields_to_check = [
    "felt",
    "cdi",
    "mmi",
    "alert",
    "nst",
    "dmin",
    "gap"
]

missing_counts = {}

for field in fields_to_check:
    missing_count = 0
    
    for record in records:
        properties = record.get("properties", {})
        
        if properties.get(field) is None:
            missing_count += 1
    
    missing_counts[field] = missing_count

In [31]:
total_records = len(records)

print(f"{'Field':<10} {'Missing':<10} {'Missing %':<10}")
print("-" * 32)

for field in fields_to_check:
    missing = missing_counts[field]
    percentage = (missing / total_records) * 100
    
    print(f"{field:<10} {missing:<10} {percentage:.2f}%")

Field      Missing    Missing % 
--------------------------------
felt       1489       83.65%
cdi        1489       83.65%
mmi        1591       89.38%
alert      1715       96.35%
nst        0          0.00%
dmin       0          0.00%
gap        0          0.00%


## 10. Event type Distribution & Extract Earthquake Event IDs

In [32]:
event_type_counts = {}

for record in records:
    event_type = record.get("properties", {}).get("type")
    
    if event_type not in event_type_counts:
        event_type_counts[event_type] = 0
    
    event_type_counts[event_type] += 1

for event_type, count in event_type_counts.items():
    print(f"{event_type}: {count}")

earthquake: 1780


In [33]:
event_ids = []

for record in records:
    event_id = record.get("id")
    
    if event_id:
        event_ids.append(event_id)

print(f"Number of extracted event IDs: {len(event_ids)}")
print("\nFirst 10 IDs:")

for event_id in event_ids[:10]:
    print(event_id)

Number of extracted event IDs: 1780

First 10 IDs:
us6000tn9r
aka2026qtvqfq
aka2026qtuxwd
us6000tn9b
us6000tn8w
aka2026qtqaig
aka2026qtpptp
aka2026qtonse
aka2026qtoknv
us6000tn80


## 11. Save Event IDs

In [34]:
ids_path = raw_data / "extracted_ids.txt"

with open(ids_path, "w", encoding="utf-8") as file:
    for event_id in event_ids:
        file.write(f"{event_id}\n")

print("Event IDs saved to:")
print(ids_path)

Event IDs saved to:
..\data\raw\extracted_ids.txt
